In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML
from scipy.stats import ttest_rel
import os
import base64

# --- 1. DATA PREPARATION ---
# Load dataset 
def get_data_path(filename):
    if os.path.exists(os.path.join('data', filename)):
        return os.path.join('data', filename)
    elif os.path.exists(os.path.join('..', 'data', filename)):
        return os.path.join('..', 'data', filename)
    else:
        raise FileNotFoundError(f"Could not find {filename} in 'data/' or '../data/'")

df1 = pd.read_excel(get_data_path('df1_EVs.xlsx'))
df2 = pd.read_excel(get_data_path('df2_EVs.xlsx'))
df3 = pd.read_excel(get_data_path('df3_EVs.xlsx'))
df4 = pd.read_excel(get_data_path('df4_EVs.xlsx'))

df_all = pd.concat([df1, df2, df3, df4], ignore_index=True)

df_all['time'] = df_all['time'].map({
    'time1': 'Baseline',
    'time2': '3 min',
    'time3': '1 hour',
    'time4': '2 hours'
})

# --- 2. DYNAMIC Wilcoxon-TEST CALCULATION FUNCTION ---
def compute_skewness_wilcoxon(df_input, alpha=0.05):
    """
    Evaluates EV size distribution asymmetry (skewness) by testing 
    paired differences between Mean Size (nm) and Median Size (nm) 
    using the Wilcoxon Signed-Rank Test.
    """
    df = df_input.copy()
    
    # 1. Normalize time column to standard keys
    time_map = {
        'baseline': 'Baseline', 'time1': 'Baseline', 'time 1': 'Baseline', '1': 'Baseline',
        '3min': '3 min', '3 min': '3 min', 'time2': '3 min', 'time 2': '3 min', '2': '3 min',
        '1hr': '1 hour', '1 hour': '1 hour', 'time3': '1 hour', 'time 3': '1 hour', '3': '1 hour',
        '2hrs': '2 hours', '2 hours': '2 hours', 'time4': '2 hours', 'time 4': '2 hours', '4': '2 hours'
    }
    
    # Find the time column name flexible to case
    time_col = next((c for c in ['time', 'Time', 'TimePoint', 'Time_Point'] if c in df.columns), 'time')
    sex_col = next((c for c in ['sex', 'Sex', 'Gender'] if c in df.columns), 'sex')
    
    df['time_clean'] = df[time_col].astype(str).str.strip().str.lower().map(lambda x: time_map.get(x, x))
    df['sex_clean'] = df[sex_col].astype(str).str.strip().str.title()

    time_order = ['Baseline', '3 min', '1 hour', '2 hours']
    results = []
    
    for t_pt in time_order:
        for s in ['Male', 'Female']:
            # Filter rows for time and sex
            sub = df[(df['time_clean'] == t_pt) & (df['sex_clean'] == s)].dropna(
                subset=['Mean Value (nm)', 'Median Value (nm)']
            )
            
            # Numeric conversion guard
            sub['Mean Value (nm)'] = pd.to_numeric(sub['Mean Value (nm)'], errors='coerce')
            sub['Median Value (nm)'] = pd.to_numeric(sub['Median Value (nm)'], errors='coerce')
            sub = sub.dropna(subset=['Mean Value (nm)', 'Median Value (nm)'])
            
            n_count = len(sub)
            
            if n_count >= 3:
                mean_vals = sub['Mean Value (nm)']
                median_vals = sub['Median Value (nm)']
                diff = mean_vals - median_vals
                
                try:
                    w_stat, p_val_w = wilcoxon(mean_vals, median_vals)
                except Exception:
                    w_stat, p_val_w = np.nan, np.nan
                
                mean_diff = diff.mean()
                median_diff = diff.median()
            else:
                w_stat, p_val_w, mean_diff, median_diff = np.nan, np.nan, np.nan, np.nan

            results.append({
                'Time Point': t_pt,
                'Sex': s,
                'N': n_count,
                'Mean Shift (Mean - Median nm)': mean_diff,
                'Median Shift (nm)': median_diff,
                'Wilcoxon_W': w_stat,
                'p_value_raw': p_val_w
            })
            
    res_df = pd.DataFrame(results)
    
    # Apply Benjamini-Hochberg FDR correction
    valid_mask = res_df['p_value_raw'].notna()
    if valid_mask.any():
        _, p_fdr, _, _ = multipletests(res_df.loc[valid_mask, 'p_value_raw'], method='fdr_bh')
        res_df.loc[valid_mask, 'p_value_FDR'] = p_fdr
        res_df['Significant'] = res_df['p_value_FDR'] < alpha
    else:
        res_df['p_value_FDR'] = np.nan
        res_df['Significant'] = False
        
    return res_df

# --- 3. INTERACTIVE CONTROLS ---
time_select = widgets.Dropdown(
    options=['All Time Points (2x2 Grid)', 'Baseline', '3 min', '1 hour', '2 hours'],
    value='All Time Points (2x2 Grid)',
    description='Time View:',
    style={'description_width': 'initial'}
)

export_btn = widgets.Button(
    description='Download Data & Stats',
    button_style='success',
    icon='download'
)

out = widgets.Output()

# --- 4. DASHBOARD RENDERER ---
def render_figure(change=None):
    with out:
        out.clear_output(wait=True)
        selected_time = time_select.value
        
        # Calculate t-tests live on current dataset
        stats_df = compute_skewness_wilcoxon(df_all)
        
        # Configure Matplotlib styles
        mpl.rcParams['svg.fonttype'] = 'none'
        plt.rcParams.update({
            'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'FreeSans', 'DejaVu Sans', 'sans-serif'],
            'font.size': 8, 'font.weight': 'bold',
            'axes.titlesize': 8.5, 'axes.titleweight': 'bold',
            'axes.labelsize': 8, 'axes.labelweight': 'bold',
            'xtick.labelsize': 7, 'ytick.labelsize': 7,
            'axes.linewidth': 1.0, 'lines.linewidth': 1.2
        })
        
        sex_markers = {'Male': 'o', 'Female': 's'}
        
        if selected_time == 'All Time Points (2x2 Grid)':
            # Render Full 2x2 Grid
            fig, axes = plt.subplots(2, 2, figsize=(6, 6), constrained_layout=True)
            time_points = ['Baseline', '3 min', '1 hour', '2 hours']
            
            for ax, t_pt in zip(axes.flat, time_points):
                subset = df_all[df_all['time'] == t_pt]
                
                sns.scatterplot(
                    data=subset, x='Median Value (nm)', y='Mean Value (nm)',
                    hue='sex', style='sex', ax=ax, s=35, 
                    palette={'Male': '0.1', 'Female': '0.6'}, markers=sex_markers,
                    edgecolor='black', linewidth=0.5
                )
                
                # Identity line (Mean = Median)
                all_vals = pd.concat([subset['Mean Value (nm)'], subset['Median Value (nm)']])
                lims = [all_vals.min() * 0.9, all_vals.max() * 1.1]
                ax.plot(lims, lims, 'k--', alpha=0.4, linewidth=1)
                ax.set_xlim(lims); ax.set_ylim(lims)
                
                # Retrieve live p-values
                p_col = next((col for col in ['p_value_raw', 'p_value_FDR', 'p-value', 'p_value_wilcoxon'] if col in stats_df.columns), None)

                if p_col:
                    m_sub = stats_df[(stats_df['Time Point'] == t_pt) & (stats_df['Sex'] == 'Male')]
                    f_sub = stats_df[(stats_df['Time Point'] == t_pt) & (stats_df['Sex'] == 'Female')]

                    m_p = m_sub[p_col].values[0] if not m_sub.empty else np.nan
                    f_p = f_sub[p_col].values[0] if not f_sub.empty else np.nan

                    m_str = f"{m_p:.2g}" if (not np.isnan(m_p) and m_p >= 0.001) else ("< 0.001" if not np.isnan(m_p) else "N/A")
                    f_str = f"{f_p:.2g}" if (not np.isnan(f_p) and f_p >= 0.001) else ("< 0.001" if not np.isnan(f_p) else "N/A")

                    ax.text(0.05, 0.95, f"Male p: {m_str}\nFemale p: {f_str}", 
                            transform=ax.transAxes, ha='left', va='top', fontsize=8, fontweight='bold',
                            bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', pad=2))
                
                ax.set_title(f'Time: {t_pt}')
                ax.set_xlabel('Median Size (nm)')
                ax.set_ylabel('Mean Size (nm)')
                
                if t_pt == '2 hours':
                    ax.legend(prop={'weight':'bold', 'size':7}, loc='lower right', frameon=True)
                else:
                    if ax.get_legend(): ax.get_legend().remove()
                    
            plt.suptitle('EV Size Skewness (Mean vs Median)', fontweight='bold')
            plt.show()
            
            display(HTML("<b>Statistical Summary (Wilcoxon Signed-Ranked Test):</b>"))
            display(stats_df)
            
        else:
            # Render Single Isolated Panel
            fig, ax = plt.subplots(figsize=(4.5, 4), constrained_layout=True)
            subset = df_all[df_all['time'] == selected_time]
            
            sns.scatterplot(
                data=subset, x='Median Value (nm)', y='Mean Value (nm)',
                hue='sex', style='sex', ax=ax, s=45, 
                palette={'Male': '0.1', 'Female': '0.6'}, markers=sex_markers,
                edgecolor='black', linewidth=0.5
            )
            
            all_vals = pd.concat([subset['Mean Value (nm)'], subset['Median Value (nm)']])
            lims = [all_vals.min() * 0.9, all_vals.max() * 1.1]
            ax.plot(lims, lims, 'k--', alpha=0.4, linewidth=1)
            ax.set_xlim(lims); ax.set_ylim(lims)
            
            p_col = next((col for col in ['p_value_raw', 'p_value_FDR', 'p-value', 'p_value_wilcoxon'] if col in stats_df.columns), None)
            
            if p_col:
                    m_sub = stats_df[(stats_df['Time Point'] == t_pt) & (stats_df['Sex'] == 'Male')]
                    f_sub = stats_df[(stats_df['Time Point'] == t_pt) & (stats_df['Sex'] == 'Female')]

                    m_p = m_sub[p_col].values[0] if not m_sub.empty else np.nan
                    f_p = f_sub[p_col].values[0] if not f_sub.empty else np.nan

                    m_str = f"{m_p:.2g}" if (not np.isnan(m_p) and m_p >= 0.001) else ("< 0.001" if not np.isnan(m_p) else "N/A")
                    f_str = f"{f_p:.2g}" if (not np.isnan(f_p) and f_p >= 0.001) else ("< 0.001" if not np.isnan(f_p) else "N/A")

                    ax.text(0.05, 0.95, f"Male p: {m_str}\nFemale p: {f_str}", 
                            transform=ax.transAxes, ha='left', va='top', fontsize=8, fontweight='bold',
                            bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', pad=2))
                
            
            ax.set_title(f'Time: {selected_time}')
            ax.set_xlabel('Median Size (nm)')
            ax.set_ylabel('Mean Size (nm)')
            ax.legend(prop={'weight':'bold', 'size':8}, loc='lower right', frameon=True)
            plt.show()
            
            display(HTML(f"<b>Statistical Summary (Wilcoxon Signed-Ranked Test) ({selected_time}):</b>"))
            display(stats_df[stats_df['Time Point'] == selected_time])

# --- 5. EXCEL EXPORT FUNCTION ---
def export_excel(b):
    with out:
        file_name = "Figure1D_EV_Size_Skewness_Data.xlsx"
        stats_df = compute_skewness_wilcoxon(df_all)
        
        with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
            stats_df.to_excel(writer, sheet_name='EV_Size_Skewness_Wilcoxon_Stats', index=False)
            df_all.to_excel(writer, sheet_name='Raw_EV_Data', index=False)
            
        # Encode file into Base64
        with open(file_name, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()

        # Trigger IMMEDIATE browser download via invisible iframe (No second click needed!)
        js_download = f"""
        <script>
            var link = document.createElement('a');
            link.href = 'data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}';
            link.download = '{file_name}';
            document.body.appendChild(link);
            link.click();
            document.body.removeChild(link);
        </script>
        <div style="color: #28a745; font-weight: bold; font-size: 12px; margin-top: 5px;">
            ✅ Downloading <i>{file_name}</i> (contains ANCOVA Stats + Post-Hoc Contrasts + Raw Data)...
        </div>
        """
        with out:
            display(HTML(js_download))

time_select.observe(render_figure, names='value')
export_btn.on_click(export_excel)

# --- 6. INITIALIZE DASHBOARD VIEW ---
display(widgets.HBox([time_select, export_btn]))
display(out)
render_figure()
